<a href="https://colab.research.google.com/github/faiq191/244107020214-machine-learning/blob/main/JS09/JS09_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup: Install Required Libraries

In [1]:
!pip install annoy faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 4.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 26.0 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520033 sha256=de7745559e00c007dc73bc217555e857df47228a3a69aca52b4d0f75956eb44f
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]


### Benchmark Execution

In [2]:
import time
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
import faiss

# collect all configuration results
all_results = []

def run_benchmark_split_metrics(n_points, dim, metric='euclidean'):
    np.random.seed(42)
    data = np.random.random((n_points, dim)).astype('float32')
    query = np.random.random((1, dim)).astype('float32')

    # --- ANNOY ---
    t0 = time.time()
    annoy_metric = 'angular' if metric == 'cosine' else 'euclidean'
    annoy_idx = AnnoyIndex(dim, annoy_metric)
    for i in range(n_points):
        annoy_idx.add_item(i, data[i])
    annoy_idx.build(10)
    annoy_build_time = time.time() - t0

    t0 = time.time()
    annoy_idx.get_nns_by_vector(query[0], 10)
    annoy_query_time = time.time() - t0

    # --- FAISS ---
    t0 = time.time()
    if metric == 'euclidean':
        faiss_idx = faiss.IndexFlatL2(dim)
    else:
        faiss_idx = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(data)
        faiss.normalize_L2(query)
    faiss_idx.add(data)
    faiss_build_time = time.time() - t0

    t0 = time.time()
    faiss_idx.search(query, 10)
    faiss_query_time = time.time() - t0

    # --- HNSW ---
    t0 = time.time()
    if metric == 'euclidean':
        hnsw_idx = faiss.IndexHNSWFlat(dim, 32)
    else:
        hnsw_idx = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
    hnsw_idx.add(data)
    hnsw_build_time = time.time() - t0

    t0 = time.time()
    hnsw_idx.search(query, 10)
    hnsw_query_time = time.time() - t0

    row_build = {
        "Number of Data/Dimensions/Metric": f"{n_points:,} / {dim}D / {metric}",
        "Metric": "Build Time",
        "ANNOY": f"{annoy_build_time:.6f}s",
        "FAISS": f"{faiss_build_time:.6f}s",
        "HNSW": f"{hnsw_build_time:.6f}s"
    }
    row_query = {
        "Number of Data/Dimensions/Metric": f"{n_points:,} / {dim}D / {metric}",
        "Metric": "Query Time",
        "ANNOY": f"{annoy_query_time:.6f}s",
        "FAISS": f"{faiss_query_time:.6f}s",
        "HNSW": f"{hnsw_query_time:.6f}s"
    }
    return [row_build, row_query]

In [3]:
all_results = []

# Run the 1,000 and 1,000,000 point test cases across 2D and 5D configurations
for n_points in [1000, 1000000]:
    for dim in [2, 5]:
        for metric in ['euclidean', 'cosine']:
            print(f"Running: {n_points:,} / {dim}D / {metric}...")
            rows = run_benchmark_split_metrics(n_points, dim, metric)
            all_results.extend(rows)

# Display the final table
df = pd.DataFrame(all_results)
display(df)

Running: 1,000 / 2D / euclidean...
Running: 1,000 / 2D / cosine...
Running: 1,000 / 5D / euclidean...
Running: 1,000 / 5D / cosine...
Running: 1,000,000 / 2D / euclidean...
Running: 1,000,000 / 2D / cosine...
Running: 1,000,000 / 5D / euclidean...
Running: 1,000,000 / 5D / cosine...


,Number of Data/Dimensions/Metric,Metric,ANNOY,FAISS,HNSW
0,"1,000 / 2D / euclidean",Build Time,0.022548s,0.000170s,0.049066s
1,"1,000 / 2D / euclidean",Query Time,0.000071s,0.000070s,0.000183s
2,"1,000 / 2D / cosine",Build Time,0.063077s,0.000149s,0.027533s
3,"1,000 / 2D / cosine",Query Time,0.000070s,0.000074s,0.000113s
4,"1,000 / 5D / euclidean",Build Time,0.021733s,0.000087s,0.031616s
5,"1,000 / 5D / euclidean",Query Time,0.000061s,0.000043s,0.000099s
6,"1,000 / 5D / cosine",Build Time,0.026848s,0.000102s,0.028026s
7,"1,000 / 5D / cosine",Query Time,0.000067s,0.000049s,0.000114s
8,"1,000,000 / 2D / euclidean",Build Time,26.608905s,0.002738s,40.697114s
9,"1,000,000 / 2D / euclidean",Query Time,0.000104s,0.007266s,0.000328s
